# Privacy & Governance Analysis

**Role:** Governance Officer

**Objective:** Assess NovaCred's credit application data for privacy risks, map findings to GDPR and EU AI Act requirements, and propose actionable governance controls.

NovaCred uses machine learning to make credit decisions and recently received a regulatory inquiry about potential discrimination. This notebook evaluates whether the company's data handling practices comply with European data protection law and AI regulation.

We focus on five areas:
1. Identification of personal data (PII) in the dataset
2. Pseudonymization demonstration
3. Mapping to specific GDPR articles
4. EU AI Act classification for credit scoring
5. Concrete governance recommendations

In [1]:
import pandas as pd
import hashlib
import json

# Load the raw dataset directly
with open('../data/raw_credit_applications.json', 'r') as f:
    raw = json.load(f)

df = pd.json_normalize(raw)
print(f"Dataset loaded: {df.shape[0]} records, {df.shape[1]} columns")
df.columns.tolist()

Dataset loaded: 502 records, 21 columns


['_id',
 'spending_behavior',
 'processing_timestamp',
 'applicant_info.full_name',
 'applicant_info.email',
 'applicant_info.ssn',
 'applicant_info.ip_address',
 'applicant_info.gender',
 'applicant_info.date_of_birth',
 'applicant_info.zip_code',
 'financials.annual_income',
 'financials.credit_history_months',
 'financials.debt_to_income',
 'financials.savings_balance',
 'decision.loan_approved',
 'decision.rejection_reason',
 'loan_purpose',
 'decision.interest_rate',
 'decision.approved_amount',
 'financials.annual_salary',
 'notes']

## 1. Personal Data Identification

Under GDPR Article 4, personal data is any information relating to an identified or identifiable natural person. We classify each field by how easily it can identify someone and the risk that creates.

In [2]:
pii_analysis = pd.DataFrame({
    'Field': [
        'full_name', 'email', 'ssn', 'ip_address',
        'date_of_birth', 'zip_code', 'gender', 'spending_behavior'
    ],
    'PII Type': [
        'Direct identifier', 'Direct identifier',
        'Sensitive national identifier', 'Online identifier',
        'Quasi-identifier', 'Quasi-identifier',
        'Special category (Art. 9)', 'Behavioral data'
    ],
    'Risk Level': [
        'High', 'High', 'Critical', 'Medium',
        'Medium', 'Medium', 'High', 'Medium-High'
    ],
    'Why It Matters': [
        'Immediately reveals identity',
        'Unique to an individual, doubles as a contact channel',
        'National ID that unequivocally identifies a person across all systems',
        'Can be linked to a household or individual, especially with timestamps',
        'Combined with zip + gender, can re-identify 87% of US population (Sweeney, 2000)',
        'Geolocation data — when paired with DOB/gender, re-identification risk is high',
        'Protected attribute under GDPR Art. 9 — requires explicit consent',
        'Spending patterns can reveal religion, health conditions, political views'
    ]
})

pii_analysis

,Field,PII Type,Risk Level,Why It Matters
0,full_name,Direct identifier,High,Immediately reveals identity
1,email,Direct identifier,High,"Unique to an individual, doubles as a contact ..."
2,ssn,Sensitive national identifier,Critical,National ID that unequivocally identifies a pe...
3,ip_address,Online identifier,Medium,"Can be linked to a household or individual, es..."
4,date_of_birth,Quasi-identifier,Medium,"Combined with zip + gender, can re-identify 87..."
5,zip_code,Quasi-identifier,Medium,Geolocation data — when paired with DOB/gender...
6,gender,Special category (Art. 9),High,Protected attribute under GDPR Art. 9 — requir...
7,spending_behavior,Behavioral data,Medium-High,"Spending patterns can reveal religion, health ..."


### Re-identification Risk

The combination of `zip_code` + `date_of_birth` + `gender` is a well-known re-identification vector. Even after removing direct identifiers like names and SSNs, these three quasi-identifiers alone can uniquely identify most individuals in a population.

Simply "deleting the name column" is not sufficient for privacy protection. NovaCred needs either proper anonymization techniques (k-anonymity, differential privacy) or strict access controls on quasi-identifiers.

Additionally, `spending_behavior` deserves special attention. Granular spending data (categories like "religious organizations" or "pharmacies") can be used to infer protected characteristics that should not influence credit decisions.

In [3]:
# Confirm that sensitive fields are stored in plain text
print("=== Sample of PII stored without protection ===\n")

sample = df.head(3)
for col in ['applicant_info.full_name', 'applicant_info.email',
            'applicant_info.ssn', 'applicant_info.ip_address']:
    if col in df.columns:
        print(f"{col}:")
        print(f"  {sample[col].tolist()}\n")

# Count records with unprotected SSNs
ssn_col = [c for c in df.columns if 'ssn' in c.lower()]
if ssn_col:
    non_null_ssns = df[ssn_col[0]].notna().sum()
    print(f"Records with SSN in plain text: {non_null_ssns} / {len(df)} ({non_null_ssns/len(df)*100:.1f}%)")

=== Sample of PII stored without protection ===

applicant_info.full_name:
  ['Jerry Smith', 'Brandon Walker', 'Scott Moore']

applicant_info.email:
  ['jerry.smith17@hotmail.com', 'brandon.walker2@yahoo.com', 'scott.moore94@mail.com']

applicant_info.ssn:
  ['596-64-4340', '425-69-4784', '370-78-5178']

applicant_info.ip_address:
  ['192.168.48.155', '10.1.102.112', '10.240.193.250']

Records with SSN in plain text: 497 / 502 (99.0%)


## 2. Pseudonymization Demonstration

Pseudonymization replaces identifying values with artificial identifiers so the data can no longer be attributed to a specific person without additional information (GDPR Art. 4(5)).

We use SHA-256 hashing with a salt. The salt prevents rainbow table attacks — pre-computed tables that map common values to their hashes.

In [4]:
def pseudonymize(value, salt="novacred_2025_audit"):
    """SHA-256 hash with salt. Returns first 12 hex characters."""
    if pd.isna(value):
        return None
    return hashlib.sha256(f"{salt}{value}".encode('utf-8')).hexdigest()[:12]

# Apply to the three most critical fields
ssn_col = [c for c in df.columns if 'ssn' in c.lower()][0]
name_col = [c for c in df.columns if 'full_name' in c.lower()][0]
email_col = [c for c in df.columns if 'email' in c.lower()][0]

df['ssn_pseudo'] = df[ssn_col].apply(pseudonymize)
df['name_pseudo'] = df[name_col].apply(pseudonymize)
df['email_pseudo'] = df[email_col].apply(pseudonymize)

# Before vs. after comparison
comparison = df[[ssn_col, 'ssn_pseudo', name_col, 'name_pseudo']].head(5)
comparison.columns = ['Original SSN', 'Pseudonymized SSN', 'Original Name', 'Pseudonymized Name']
comparison

,Original SSN,Pseudonymized SSN,Original Name,Pseudonymized Name
0,596-64-4340,c5fac20022d2,Jerry Smith,f9271ba16b39
1,425-69-4784,a96bd6c86301,Brandon Walker,a4a6a2ccb861
2,370-78-5178,833cb8435189,Scott Moore,762fd86e6ce3
3,194-35-1833,d71d0d3f7e6e,Thomas Lee,bee84b0e3b69
4,480-41-2475,542360cfe634,Brian Rodriguez,6ad8aaa8ba40


### What pseudonymization achieves and doesn't achieve

**Achieves:**
- Prevents casual identification: anyone viewing the dataset cannot read names or SSNs
- Deterministic: the same SSN always maps to the same hash, so duplicate detection and joins still work
- One-way: you cannot reverse the hash to recover the original value

**Does not achieve:**
- **True anonymization.** Under GDPR, pseudonymized data is still personal data because re-identification is possible if the salt is known. Full anonymization would require techniques like k-anonymity or differential privacy.
- **Protection against the data controller.** NovaCred itself could still re-identify records since it holds the original data. Pseudonymization protects against external breaches, not internal misuse.

This distinction matters for compliance because pseudonymized data remains subject to GDPR requirements, while truly anonymous data does not.

## 3. GDPR Requirements Mapping

We assess NovaCred's data practices against the GDPR provisions most relevant to automated credit scoring.

In [5]:
gdpr_mapping = pd.DataFrame({
    'Article': [
        'Art. 5(1)(c): Data Minimization',
        'Art. 5(1)(e): Storage Limitation',
        'Art. 6(1): Lawful Basis',
        'Art. 9: Special Categories',
        'Art. 17: Right to Erasure',
        'Art. 22: Automated Decisions',
        'Art. 25: Privacy by Design',
        'Art. 35: Impact Assessment'
    ],
    'What It Requires': [
        'Collect only what is adequate, relevant, and necessary',
        'Keep personal data only as long as needed',
        'Have a legal basis: consent, contract, or legitimate interest',
        'Special data (gender, health) needs explicit consent or legal exception',
        'Delete personal data on request when no longer needed',
        'Right not to be subject to purely automated decisions with legal effects',
        'Build data protection into systems from the start',
        'Conduct impact assessment before high-risk processing'
    ],
    'NovaCred Gap': [
        'IP addresses and granular spending categories are not necessary for credit assessment',
        'No retention policy : data stored indefinitely with no documented expiry',
        'No consent field exists in the dataset : lawful basis is undocumented',
        'Gender is collected and influences decisions without documented Art. 9 exception',
        'SSNs in plain text with no deletion mechanism : erasure requests cannot be fulfilled cleanly',
        'All credit decisions are automated : no human review process documented',
        'PII stored unprotected, no encryption or access controls evident in the data',
        'No DPIA conducted despite high-risk automated credit scoring'
    ],
    'Severity': ['Medium', 'High', 'High', 'High', 'Critical', 'Critical', 'High', 'High']
})

gdpr_mapping.style.set_properties(**{'text-align': 'left'})

,Article,What It Requires,NovaCred Gap,Severity
0,Art. 5(1)(c): Data Minimization,"Collect only what is adequate, relevant, and necessary",IP addresses and granular spending categories are not necessary for credit assessment,Medium
1,Art. 5(1)(e): Storage Limitation,Keep personal data only as long as needed,No retention policy : data stored indefinitely with no documented expiry,High
2,Art. 6(1): Lawful Basis,"Have a legal basis: consent, contract, or legitimate interest",No consent field exists in the dataset : lawful basis is undocumented,High
3,Art. 9: Special Categories,"Special data (gender, health) needs explicit consent or legal exception",Gender is collected and influences decisions without documented Art. 9 exception,High
4,Art. 17: Right to Erasure,Delete personal data on request when no longer needed,SSNs in plain text with no deletion mechanism : erasure requests cannot be fulfilled cleanly,Critical
5,Art. 22: Automated Decisions,Right not to be subject to purely automated decisions with legal effects,All credit decisions are automated : no human review process documented,Critical
6,Art. 25: Privacy by Design,Build data protection into systems from the start,"PII stored unprotected, no encryption or access controls evident in the data",High
7,Art. 35: Impact Assessment,Conduct impact assessment before high-risk processing,No DPIA conducted despite high-risk automated credit scoring,High


### The Two Critical Failures

**Art. 22: Automated Decision-Making:** Credit approval/rejection is a decision that "produces legal effects" on the applicant (they either get a loan or they don't). Under Art. 22, NovaCred must either obtain explicit consent for automated processing, demonstrate contractual necessity, or implement safeguards including the right to human review. The dataset shows no mechanism for any of these.

**Art. 17: Right to Erasure:** If an applicant requests that NovaCred delete their data, the company must be able to locate and remove all their personal information. With SSNs, names, and emails stored in plain text across what may be multiple systems, this is operationally difficult and legally risky. Pseudonymization at the point of collection would significantly reduce this burden.

## 4. EU AI Act Classification

Under Annex III, Section 5(b) of the EU AI Act, AI systems used to evaluate creditworthiness or establish credit scores are classified as <u>high-risk</u>. This applies automatically to any AI system making or influencing credit decisions.

### Compliance obligations for high-risk systems:

| Requirement | AI Act Article | NovaCred Status |
|------------|---------------|-----------------|
| Risk management system | Art. 9 | Not implemented |
| Data quality & governance | Art. 10 | Not adequately addressed |
| Technical documentation | Art. 11 | Not implemented |
| Audit trail / logging | Art. 12 | No decision logging exists |
| Transparency to users | Art. 13 | Applicants not informed of AI use |
| Human oversight mechanism | Art. 14 | No human review process |
| Accuracy & robustness | Art. 15 | No performance metrics available |

NovaCred cannot legally deploy this system in the EU without addressing these gaps. Any evidence of discriminatory outcomes in the credit decisions would compound the compliance failure, as the AI Act explicitly requires that high-risk systems do not produce biased results across protected groups.

## 5. Governance Recommendations

### 5.1 Decision Audit Trail
Log every credit decision with: timestamp, applicant ID, features used, model version, output (approve/deny), and confidence score. This satisfies AI Act Art. 12 and enables responses to individual complaints or regulatory audits.

### 5.2 Human-in-the-Loop
Flag borderline cases for manual review — any application where the model's confidence falls within 10% of the approval threshold. A designated reviewer must have authority to override the model. Required by GDPR Art. 22 and AI Act Art. 14.

### 5.3 Informed Consent & Transparency
Before collecting data, inform applicants that: (a) an AI system will evaluate their application, (b) what data is collected and why, (c) they have the right to request human review. Add a `consent_given` field to the application schema. No such field currently exists.

### 5.4 Data Retention Policy
Define retention periods: application data kept for 7 years (aligned with financial record-keeping norms), then fully anonymized or deleted. Currently no policy exists and data is retained indefinitely.

### 5.5 Minimize Unnecessary Data
Remove or anonymize `ip_address` after initial fraud screening. Aggregate `spending_behavior` into broad categories (essentials, discretionary, financial) rather than storing granular data that can reveal protected characteristics.

### 5.6 Quarterly Bias Monitoring
Compute disparate impact ratios for gender and age at regular intervals. Any DI below 0.8 triggers automatic review and model retraining. This operationalizes the non-discrimination requirement of the AI Act.

## 6. Conclusion

NovaCred's credit scoring system has significant governance gaps:

- 7 PII fields stored without protection, including SSNs in plain text
- 8 GDPR articles where compliance gaps were identified, with critical failures in automated decision-making and right to erasure
- All 7 AI Act requirements for high-risk systems unmet
- Zero governance infrastructure: no consent tracking, no retention policy, no audit trail, no human oversight

These findings indicate that NovaCred's system would not survive a regulatory audit in its current form. The recommendations in Section 5 provide a prioritized roadmap for remediation, starting with the two critical items: implementing human oversight for credit decisions and establishing an erasure mechanism for personal data.